# Data Agent (LangGraph): prices, cleaning, risk metrics and data checks

**Role in the team:** the Data Agent runs first. It downloads the ETF prices, cleans them, computes returns, volatility and drawdowns, and checks for data errors. The other four agents only read what it saves.

| Agent | What it uses from the Data Agent |
|---|---|
| 2. Technical | `prices_clean.csv` (adjusted prices) for moving averages and momentum |
| 3. Macro / News | `metrics.csv` (recent returns by theme) to link news to market moves |
| 4. Risk | `metrics.csv` (volatility, drawdowns, leverage factor, currency, liquidity) and `data_issues.csv` |
| 5. Portfolio Manager | `returns.csv`, `close_clean.csv` (raw close, to size orders) and the data report |

The notebook follows the same steps as the LangGraph demo notebook: tools, LLM, agent node, graph, run, inspect.

**Rule:** the LLM never produces numbers itself. All figures come from Python tools reading the downloaded data, and the agent is instructed to report only what the tools return.

## 1. Install dependencies
Run once. `download_etfs.py` must be in the same folder as this notebook.

In [1]:
pip install -q -U langgraph langchain langchain-openai python-dotenv yfinance pandas requests

Note: you may need to restart the kernel to use updated packages.


## 2. Configure your LLM
Same `.env` file as the demo notebook: `BASE_URL`, `MODEL`, `API_KEY`. Do not commit it.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["BASE_URL"]
MODEL = os.environ["MODEL"]
API_KEY = os.environ["API_KEY"]

## 3. Imports
We reuse the logic of `download_etfs.py` (ETF list, OpenFIGI tickers, Yahoo download, manual corrections, leverage factors, currencies), so there is a single source of truth for the data.

Corrections applied automatically by `download_etfs.apply_corrections` (see `SPLITS` and `BAD_PRICES` there):
- EURO BANKS: 80:1 split on 2026-10-26 (prices before divided by 80)
- PALLADIUM LVG: 4:1 split on 2022-12-19 (prices before divided by 4)
- 24/10/2025, zero-volume wrong prices replaced by the previous day: WORLD MIN VOL, ROBOTICS, NASDAQ 3X, COPPER LVG, PALLADIUM LVG, S&P 500 SHORT
- COPPER LVG on 2025-07-31 kept on purpose (genuine move)

In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

import download_etfs as dl  # ETFS, full(), update(), apply_corrections(), PRICES_FILE, ...

DATA_DIR = dl.DATA_DIR
PRICES_FILE = dl.PRICES_FILE

## 4. Settings (documented assumptions) A REVOIR AVEC LA TEAM
These thresholds are choices, not universal rules. They are explained in the report (data quality section).

In [6]:
TRADING_DAYS = 252          # trading days per year, used to annualise volatility
FFILL_LIMIT = 5             # carry the last price forward for at most 5 trading days
LONG_GAP_DAYS = 5           # a gap longer than one trading week is a data problem, not a holiday
EXTREME_MOVE = 0.20         # daily moves above 20% are flagged for manual review
SUSPICIOUS_MOVE = 0.08      # a move above 8% with zero volume that reverses next day = likely bad price
SHORT_HISTORY_DAYS = 252    # less than ~1 year of data: long-term metrics are not reliable

WINDOWS = {"1w": 5, "1m": 21, "3m": 63, "6m": 126, "1y": 252}   # in trading days


# Output files shared with the other agents
OUT = {
    "prices_clean": DATA_DIR / "prices_clean.csv",   # adjusted close, wide (date x ETF)
    "close_clean": DATA_DIR / "close_clean.csv",     # raw close, wide (what Belfius uses for orders)
    "returns": DATA_DIR / "returns.csv",             # daily returns, wide
    "metrics": DATA_DIR / "metrics.csv",             # one row per ETF
    "issues": DATA_DIR / "data_issues.csv",          # one row per detected issue
    "report": DATA_DIR / "data_report.md",           # the agent's written summary
}

## 5. Data functions (plain Python, no LLM)
Each step is a normal function, so it can be tested on its own and reused by teammates.

In [7]:
def load_prices() -> pd.DataFrame:
    """Long table from download_etfs.py, with the manual corrections applied."""
    df = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    df = df.drop(columns=["return", "leverage", "currency"], errors="ignore")
    df = dl.add_static_columns(dl.apply_corrections(df))   # + leverage and currency columns
    return df.sort_values(["code", "date"]).reset_index(drop=True)


def clean_prices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Wide tables (date x ETF) on a common calendar.
    Gaps are forward-filled for at most FFILL_LIMIT days; nothing is filled before an ETF's launch
    and nothing is interpolated (that would use future prices)."""
    adj = df.pivot(index="date", columns="code", values="adj_close").sort_index()
    close = df.pivot(index="date", columns="code", values="close").sort_index()
    return adj.ffill(limit=FFILL_LIMIT), close.ffill(limit=FFILL_LIMIT)


def max_drawdown(prices: pd.Series) -> float:
    """Largest fall from a previous peak (negative number, e.g. -0.35 = -35%)."""
    p = prices.dropna()
    if p.empty:
        return np.nan
    return float((p / p.cummax() - 1).min())


def compute_metrics(adj: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """One row per ETF: returns, volatility, drawdowns, liquidity, leverage flag."""
    returns = adj.pct_change(fill_method=None)
    info = df.drop_duplicates("code").set_index("code")[["isin", "yahoo", "theme", "leverage", "currency"]]
    rows = []
    for code in adj.columns:
        p = adj[code].dropna()
        r = returns[code].dropna()
        vol = df[df["code"] == code].set_index("date")["volume"]
        row = {
            "code": code,
            "theme": info.loc[code, "theme"],
            "leverage": int(info.loc[code, "leverage"]),
            "leveraged_or_inverse": int(info.loc[code, "leverage"]) != 1,
            "currency": info.loc[code, "currency"],
            "first_date": p.index[0].date(),
            "last_date": p.index[-1].date(),
            "history_days": len(p),
            "last_adj_close": p.iloc[-1],
        }
        for name, n in WINDOWS.items():   # period returns (NaN if history too short)
            row[f"ret_{name}"] = p.iloc[-1] / p.iloc[-n - 1] - 1 if len(p) > n else np.nan
        for name in ["1m", "3m", "1y"]:   # annualised volatility
            n = WINDOWS[name]
            row[f"vol_{name}"] = r.iloc[-n:].std() * np.sqrt(TRADING_DAYS) if len(r) >= n else np.nan
        row["max_drawdown_all"] = max_drawdown(p)
        row["max_drawdown_1y"] = max_drawdown(p.iloc[-WINDOWS["1y"]:]) if len(p) > WINDOWS["1y"] else np.nan
        row["current_drawdown"] = p.iloc[-1] / p.max() - 1
        row["avg_volume_1m"] = vol.iloc[-WINDOWS["1m"]:].mean()
        row["zero_volume_days_1m"] = int((vol.iloc[-WINDOWS["1m"]:] == 0).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("code")


def detect_issues(df: pd.DataFrame) -> pd.DataFrame:
    """List of data problems, one row per issue. Nothing is changed here: issues are for review."""
    issues = []
    add = lambda code, date, kind, detail: issues.append(
        {"code": code, "date": date, "issue": kind, "detail": detail})
    global_start = df["date"].min()
    global_end = df["date"].max()

    for code, g in df.groupby("code"):
        g = g.set_index("date").sort_index()
        leverage = int(g["leverage"].iloc[0])
        currency = g["currency"].iloc[0]
        ret = g["adj_close"].pct_change(fill_method=None)

        if currency != dl.EXPECTED_CURRENCY:
            add(code, None, "currency_not_eur",
                f"currency '{currency or 'unknown'}' (run 'python download_etfs.py currency' if unknown)")

        if (g["close"] <= 0).any():
            add(code, None, "non_positive_price", f"{int((g['close'] <= 0).sum())} rows with price <= 0")
        if len(g) < SHORT_HISTORY_DAYS:
            add(code, g.index[0].date(), "short_history", f"only {len(g)} trading days of data")
        elif g.index[0] > global_start + pd.Timedelta(days=30):
            add(code, g.index[0].date(), "late_start", "starts later than the other ETFs")
        if g.index[-1] < global_end:
            add(code, g.index[-1].date(), "stale", f"last price older than {global_end.date()}")

        # gaps vs the weekday calendar (holidays give 1-3 day gaps, which is normal)
        cal = pd.bdate_range(g.index[0], g.index[-1])
        missing = ~cal.isin(g.index)
        if missing.any():
            runs = pd.Series(missing).groupby((~pd.Series(missing)).cumsum()).sum()
            if runs.max() > LONG_GAP_DAYS:
                add(code, None, "long_gap", f"longest gap {int(runs.max())} trading days")

        # extreme moves (real for leveraged ETFs sometimes: review, do not auto-correct)
        for d, x in ret[ret.abs() > EXTREME_MOVE].items():
            note = f" (leverage {leverage:+d}x: may be genuine)" if leverage != 1 else ""
            add(code, d.date(), "extreme_move", f"{x:+.1%} in one day{note}")

        # likely bad price: big move with zero volume, reversed the next day
        nxt = ret.shift(-1)
        bad = (ret.abs() > SUSPICIOUS_MOVE) & (g["volume"] == 0) & (np.sign(nxt) == -np.sign(ret)) \
              & (nxt.abs() > 0.5 * ret.abs())
        for d in ret[bad].index:
            add(code, d.date(), "suspicious_print", f"{ret[d]:+.1%} with zero volume, then {nxt[d]:+.1%}")

    return pd.DataFrame(issues, columns=["code", "date", "issue", "detail"])

### Quick test without the LLM
Runs the whole pipeline on the current `data/prices.csv`.

In [8]:
_df = load_prices()
_adj, _close = clean_prices(_df)
_metrics = compute_metrics(_adj, _df)
_issues = detect_issues(_df)
print(f"{_metrics.shape[0]} ETFs, {_adj.shape[0]} dates, {len(_issues)} issues")
_metrics[["theme", "last_date", "ret_1m", "vol_1y", "max_drawdown_all"]].head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/prices.csv'

## 6. Define the tools
Tools return short JSON text so the LLM sees compact, exact figures. Heavy tables are saved to files for the other agents instead of being sent to the LLM.

In [ ]:
def _json(obj) -> str:
    return json.dumps(obj, default=str, indent=1)


def _round(d: dict) -> dict:
    return {k: (round(v, 4) if isinstance(v, (float, np.floating)) and not np.isnan(v) else v) for k, v in d.items()}


@tool
def get_data_status() -> str:
    """Check whether the price file exists, its date range, the number of ETFs, and today's date.
    Use it first to decide if prices must be downloaded or updated."""
    today = pd.Timestamp.today().normalize()
    last_business_day = today - pd.offsets.BDay(1)   # last full trading day (closing prices available)
    if not PRICES_FILE.exists():
        return _json({"price_file_exists": False, "today": today.date()})
    df = pd.read_csv(PRICES_FILE, usecols=["date", "code"], parse_dates=["date"])
    return _json({
        "price_file_exists": True,
        "first_date": df["date"].min().date(),
        "last_date": df["date"].max().date(),
        "n_etfs": df["code"].nunique(),
        "today": today.date(),
        "last_business_day": last_business_day.date(),
        "up_to_date": df["date"].max() >= last_business_day,
    })


@tool
def refresh_prices(mode: str) -> str:
    """Download prices from Yahoo Finance. mode='full' downloads the whole history (use only if no
    price file exists), mode='update' appends the missing days (use if the file is not up to date)."""
    if mode not in ("full", "update"):
        return "mode must be 'full' or 'update'"
    try:
        dl.full() if mode == "full" else dl.update()
    except SystemExit as e:          # download_etfs uses sys.exit for clear error messages
        return f"Download stopped: {e}"
    except Exception as e:
        return f"Download failed: {e}"
    df = pd.read_csv(PRICES_FILE, usecols=["date", "code"], parse_dates=["date"])
    return _json({"status": "ok", "mode": mode, "last_date": df["date"].max().date(),
                  "n_etfs": df["code"].nunique()})


@tool
def build_clean_dataset() -> str:
    """Clean the prices (corrections + limited forward fill), compute returns, volatility and drawdowns,
    detect data issues, and save all files for the other agents. Returns a short summary."""
    df = load_prices()
    adj, close = clean_prices(df)
    returns = adj.pct_change(fill_method=None)
    metrics = compute_metrics(adj, df)
    issues = detect_issues(df)

    adj.to_csv(OUT["prices_clean"])
    close.to_csv(OUT["close_clean"])
    returns.to_csv(OUT["returns"])
    metrics.to_csv(OUT["metrics"])
    issues.to_csv(OUT["issues"], index=False)

    return _json({
        "n_etfs": metrics.shape[0],
        "date_range": [adj.index.min().date(), adj.index.max().date()],
        "files_saved": {k: str(v) for k, v in OUT.items() if k != "report"},
        "issues_by_type": issues["issue"].value_counts().to_dict(),
        "etfs_with_short_history": issues.loc[issues["issue"] == "short_history", "code"].tolist(),
        "leverage_factors": metrics.loc[metrics["leveraged_or_inverse"], "leverage"].to_dict(),
        "currencies": metrics["currency"].replace("", "unknown").value_counts().to_dict(),
    })


@tool
def list_data_issues(issue_type: str = "all") -> str:
    """List detected data issues. issue_type: 'all', 'extreme_move', 'suspicious_print', 'long_gap',
    'short_history', 'late_start', 'stale', 'non_positive_price', 'currency_not_eur'. Run build_clean_dataset first."""
    if not OUT["issues"].exists():
        return "No issues file yet: run build_clean_dataset first."
    issues = pd.read_csv(OUT["issues"])
    if issue_type != "all":
        issues = issues[issues["issue"] == issue_type]
    return _json(issues.to_dict(orient="records")) if len(issues) else "No issues of this type."


@tool
def get_risk_overview(sort_by: str = "vol_1y", top_n: int = 10) -> str:
    """Rank ETFs by a metric from metrics.csv, e.g. 'vol_1y', 'max_drawdown_all', 'ret_1m', 'ret_3m',
    'current_drawdown'. Returns the top_n rows (highest first; for drawdowns, the deepest first)."""
    if not OUT["metrics"].exists():
        return "No metrics file yet: run build_clean_dataset first."
    m = pd.read_csv(OUT["metrics"], index_col="code")
    if sort_by not in m.columns:
        return f"Unknown metric. Available: {', '.join(m.columns)}"
    ascending = "drawdown" in sort_by
    cols = ["theme", sort_by, "vol_1y", "max_drawdown_all", "history_days"]
    cols = list(dict.fromkeys(cols))
    top = m.sort_values(sort_by, ascending=ascending).head(top_n)[cols]
    return _json({code: _round(row) for code, row in top.to_dict(orient="index").items()})


@tool
def get_etf_metrics(code: str) -> str:
    """All metrics and data issues for one ETF, by its short code (e.g. 'WORLD', 'GOLD', 'NASDAQ 3X')."""
    if not OUT["metrics"].exists():
        return "No metrics file yet: run build_clean_dataset first."
    m = pd.read_csv(OUT["metrics"], index_col="code")
    if code not in m.index:
        return f"Unknown code. Available: {', '.join(m.index)}"
    issues = pd.read_csv(OUT["issues"])
    return _json({"metrics": _round(m.loc[code].to_dict()),
                  "issues": issues[issues["code"] == code].to_dict(orient="records")})


tools = [get_data_status, refresh_prices, build_clean_dataset, list_data_issues, get_risk_overview, get_etf_metrics]
print(get_data_status.invoke({}))

## 7. Connect the LLM to the tools
The system prompt sets the agent's job and the rule against inventing numbers.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

llm = ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are the Data Agent of an ETF portfolio team. You run before the Technical, Macro/News,
Risk and Portfolio Manager agents, who rely on your files.

Your job:
1. Call get_data_status. If there is no price file, call refresh_prices('full'). If the file is not up to date,
   call refresh_prices('update'). If a download fails, say so and continue with the existing file.
2. Call build_clean_dataset to clean the data, compute metrics and save the files.
3. Review data quality with list_data_issues: explain each suspicious_print and extreme_move, and say whether it
   looks like a data error or may be a genuine move (leveraged/inverse products can move a lot).
4. Use get_risk_overview to point out the most volatile ETFs and the deepest drawdowns.
5. Write a short data report for the other agents: date range, number of ETFs, issues to watch,
   ETFs with short history, leveraged/inverse ETFs and their leverage factor, any ETF not quoted in EUR,
   highest-risk ETFs.

Rules: every number you write must come from a tool result. Never estimate, round up or invent figures or dates.
If information is missing, say it is missing. You do not recommend trades: that is the other agents' job."""

## 8. Define the agent node

In [ ]:
def agent(state: MessagesState):
    messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

## 9. Assemble and compile the graph
Same loop as the demo: the agent calls tools until it has what it needs, then writes the report.

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
graph.add_edge("tools", "agent")

data_agent_app = graph.compile()

## 10. Visualize the flow

In [ ]:
from IPython.display import Image, display

try:
    display(Image(data_agent_app.get_graph().draw_mermaid_png()))
except Exception:
    print(data_agent_app.get_graph().draw_mermaid())

## 11. Run the Data Agent

In [ ]:
result = data_agent_app.invoke(
    {"messages": [{"role": "user", "content": "Prepare the market data for today's analysis and write the data report."}]},
    {"recursion_limit": 30},
)
data_report = result["messages"][-1].content
OUT["report"].write_text(data_report, encoding="utf-8")
print(data_report)

## 12. Inspect the agentic loop
Check which tools the agent called and in which order. Keep this trace: the jury wants to see the AI reasoning.

In [ ]:
for i, message in enumerate(result["messages"], start=1):
    print(f"\nStep {i} — {message.type}")
    if getattr(message, "tool_calls", None):
        print("Tool calls:", [(c["name"], c["args"]) for c in message.tool_calls])
    if message.content:
        print("Content:", message.content[:800])

## 13. Check the report against the data
The agent is told not to invent numbers, but LLMs can still make mistakes. A human checks the report against the tables below before the other agents use it.

In [ ]:
metrics = pd.read_csv(OUT["metrics"], index_col="code")
issues = pd.read_csv(OUT["issues"])
display(issues)
display(metrics.sort_values("vol_1y", ascending=False)[["theme", "vol_1y", "max_drawdown_all", "ret_1m", "history_days"]])

## 14. Hand-off to the other agents
### 14a. Data contract
The other agents load the Data Agent's outputs with one function. Agree on these names as a team: changing a column name breaks the others.

In [ ]:
def load_data_package() -> dict:
    """Everything the other agents need, as pandas objects."""
    return {
        "prices": pd.read_csv(OUT["prices_clean"], index_col="date", parse_dates=True),  # adjusted close
        "close": pd.read_csv(OUT["close_clean"], index_col="date", parse_dates=True),    # raw close (orders)
        "returns": pd.read_csv(OUT["returns"], index_col="date", parse_dates=True),
        "metrics": pd.read_csv(OUT["metrics"], index_col="code"),
        "issues": pd.read_csv(OUT["issues"]),
        "report": OUT["report"].read_text(encoding="utf-8") if OUT["report"].exists() else "",
    }

pkg = load_data_package()
{k: (v.shape if hasattr(v, "shape") else f"{len(v)} characters") for k, v in pkg.items()}

### 14b. Plug the Data Agent into the team graph
The team graph has one shared state. The Data Agent is the first node: it fills `data_report` and `data_files`, then hands over.
The other nodes below are **placeholders** for teammates (Technical, Macro/News + Risk, Portfolio Manager with human approval). Replace each one with the real agent when it is ready.

In [ ]:
from typing import TypedDict


class TeamState(MessagesState):
    data_report: str          # written by the Data Agent
    data_files: dict          # paths of the shared files
    technical_scores: dict    # Technical Agent: {code: "buy" / "hold" / "avoid"}
    macro_view: str           # Macro/News Agent
    risk_rules: dict          # Risk Agent: e.g. max weight per ETF, cap on leveraged ETFs
    proposal: str             # Portfolio Manager: proposed orders with reasoning
    human_decision: str       # "approved" / "rejected" + justification


def data_node(state: TeamState):
    out = data_agent_app.invoke(
        {"messages": [{"role": "user", "content": "Prepare the market data for today's analysis and write the data report."}]},
        {"recursion_limit": 30},
    )
    report = out["messages"][-1].content
    OUT["report"].write_text(report, encoding="utf-8")
    return {"data_report": report, "data_files": {k: str(v) for k, v in OUT.items()}}


# ---- Placeholders: to be replaced by each teammate's agent ----
def technical_node(state: TeamState):
    return {"technical_scores": {}}      # TODO (Person 2): read load_data_package()["prices"]

def macro_risk_node(state: TeamState):
    return {"macro_view": "", "risk_rules": {}}   # TODO (Person 3): read metrics + issues

def portfolio_manager_node(state: TeamState):
    return {"proposal": ""}              # TODO (Person 4): combine all, then human approval (interrupt)


team = StateGraph(TeamState)
team.add_node("data", data_node)
team.add_node("technical", technical_node)
team.add_node("macro_risk", macro_risk_node)
team.add_node("portfolio_manager", portfolio_manager_node)
team.add_edge(START, "data")
team.add_edge("data", "technical")
team.add_edge("technical", "macro_risk")
team.add_edge("macro_risk", "portfolio_manager")
team.add_edge("portfolio_manager", END)
team_app = team.compile()

try:
    display(Image(team_app.get_graph().draw_mermaid_png()))
except Exception:
    print(team_app.get_graph().draw_mermaid())

## Takeaways
- **Deterministic first:** download, cleaning and metrics are plain Python functions, tested without the LLM.
- **Agentic on top:** the LLM decides whether to download or update, reviews the data issues and writes the report.
- **No invented numbers:** every figure comes from a tool; a human checks the report (step 13).
- **Shared contract:** the other agents read the saved files through `load_data_package()`.

*Notes:* thresholds in section 4 are documented assumptions. Data comes from Yahoo Finance and may contain errors; manual corrections live in `download_etfs.py`. API calls may incur costs.